# Hybrid RAG (Sparse + Dense + Rerank) – Colab T4 GPU
Clean, self-contained notebook. All heavy models (embeddings + cross-encoder) run on the **T4 GPU**. Ollama runs the local LLM.

## 0. Environment & Installs
One install cell. Pins numpy/scipy to avoid the common Colab AttributeError.

In [1]:
# Check GPU first
!nvidia-smi

Sun Nov  1 09:50:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# 1. Install uv itself (only needed once per runtime)
%pip install -q uv

In [4]:

# -------------------------------------------------
# 2. Core numerical stack (Python 3.13 compatible)
# -------------------------------------------------
!uv pip install --system "numpy==2.1.3" "scipy==1.14.1"

# -------------------------------------------------
# 3. LangChain ecosystem
# -------------------------------------------------
!uv pip install --system \
  langchain \
  langchain-community \
  langchain-core \
  langchain-text-splitters \
  langchain-ollama \
  langchain-huggingface

# -------------------------------------------------
# 4. Retrieval & embedding stack
# -------------------------------------------------
!uv pip install --system \
  rank-bm25 \
  sentence-transformers \
  chromadb

# -------------------------------------------------
# 5. Document loaders + evaluation tools
# -------------------------------------------------
!uv pip install --system \
  pypdf \
  docx2txt \
  pandas \
  scikit-learn \
  tqdm \
  rouge-score \
  nltk \
  rapidfuzz \
  unstructured-client \
  langchain-unstructured

Using Python 3.13.16 environment at: /usr
Resolved 2 packages in 181ms
Prepared 1 package in 1.94s
Uninstalled 1 package in 168ms
Installed 1 package in 83ms
 - scipy==1.16.3
 + scipy==1.14.1
Using Python 3.13.16 environment at: /usr
Checked 6 packages in 312ms
Using Python 3.13.16 environment at: /usr
Checked 3 packages in 287ms
Using Python 3.13.16 environment at: /usr
Checked 10 packages in 129ms


In [14]:
!uv pip install --system langchain-classic

Using Python 3.13.16 environment at: /usr
Checked 1 package in 138ms


> **After the install finishes → Runtime → Restart runtime**  
> Then run all cells below this point (skip the install cell).

## 1. Start Ollama (local LLM on Colab)
Minimal, reliable install + serve + pull. No redundant steps.

In [7]:
!apt-get update -qq && apt-get install -y -qq pciutils lshw zstd > /dev/null

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


In [8]:
import subprocess, time, shutil

# Install tools so Ollama can detect the GPU


def start_ollama(model: str = "qwen2.5:3b"):
    if not shutil.which("ollama"):
        print("Installing Ollama...")
        !curl -fsSL https://ollama.com/install.sh | sh
    else:
        print("Ollama already installed.")

    # Start server if needed
    try:
        import requests
        requests.get("http://127.0.0.1:11434", timeout=2)
        print("Ollama server already running.")
    except Exception:
        print("Starting Ollama server...")
        subprocess.Popen(
            ["ollama", "serve"],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
        )
        time.sleep(6)

    print(f"Pulling model: {model}")
    !ollama pull {model}
    print("✅ Ollama ready")

start_ollama("qwen2.5:3b")

Ollama already installed.
Ollama server already running.
Pulling model: qwen2.5:3b

✅ Ollama ready


## 2. Load & Chunk Documents
Put your files under `/content/TEST_RAG` (or set `RAG_DATA_DIR`).

In [9]:
import os
from pathlib import Path
from langchain_unstructured import UnstructuredLoader
from langchain_core.documents import Document

# Paste your Unstructured API key here
os.environ["UNSTRUCTURED_API_KEY"] = "jjC34igUp0EImxYaVq456MIfcNvebd"

DATA_DIR = Path(os.environ.get("RAG_DATA_DIR", "/content/TEST_RAG"))
SUPPORTED = {".pdf", ".docx", ".doc", ".pptx", ".html", ".txt", ".md", ".eml"}

chunks: list[Document] = []

for path in sorted(DATA_DIR.rglob("*")):
    if not path.is_file() or path.suffix.lower() not in SUPPORTED:
        continue
    print(f"Loading {path.name} ...")
    try:
        loader = UnstructuredLoader(
            file_path=str(path),
            partition_via_api=True,
            api_key=os.environ["UNSTRUCTURED_API_KEY"],
            strategy="hi_res",
            chunking_strategy="by_title",
            max_characters=2000,
            new_after_n_chars=1800,
            overlap=200,
            overlap_all=True,
        )
        docs = loader.load()
        for d in docs:
            d.metadata["source"] = str(path)
        chunks.extend(docs)
        print(f"  → {len(docs)} chunks")
    except Exception as e:
        print(f"  ⚠️  skipped {path.name}: {type(e).__name__}: {e}")

for i, c in enumerate(chunks):
    c.metadata["chunk_id"] = i

print(f"\n✅ Total chunks: {len(chunks)}")

Loading TEST_HISTORY_RAG_PDF.pdf ...
  → 46 chunks

✅ Total chunks: 46


## 3. Retrievers (Sparse + Dense + Hybrid + Rerank)
All embedding / cross-encoder models forced onto the **T4 GPU**.

In [15]:
import torch
from langchain_community.retrievers import BM25Retriever
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.runnables import RunnableLambda
from sentence_transformers import CrossEncoder
from langchain_classic.retrievers import EnsembleRetriever


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

TOP_K = 10
RERANK_TOP_N = 5

# ---- Dense retriever (GPU) ----
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={"device": DEVICE},
    encode_kwargs={"normalize_embeddings": True},
)
print("Building Chroma index...")
vectorstore = Chroma.from_documents(chunks, embedding_model)
dense_retriever = vectorstore.as_retriever(search_kwargs={"k": TOP_K})

# ---- Sparse retriever ----
bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = TOP_K

# ---- Hybrid ----
hybrid_retriever = (
    EnsembleRetriever(
        retrievers=[bm25_retriever, dense_retriever],
        weights=[0.3, 0.7],
    )
    | RunnableLambda(lambda docs: docs[:TOP_K])
)

# ---- Cross-encoder reranker (GPU) ----
print("Loading cross-encoder on", DEVICE, "...")
cross_encoder = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2",
    device=DEVICE,
)

def hybrid_reranked_retrieve(query: str, top_n: int = RERANK_TOP_N):
    docs = hybrid_retriever.invoke(query)
    if not docs:
        return []
    pairs = [[query, d.page_content] for d in docs]
    scores = cross_encoder.predict(pairs)
    ranked = sorted(zip(docs, scores), key=lambda x: float(x[1]), reverse=True)
    return [doc for doc, _ in ranked[:top_n]]

class FnRetriever:
    def __init__(self, fn):
        self.fn = fn
    def invoke(self, query: str):
        return self.fn(query)

RETRIEVERS = {
    "sparse":          bm25_retriever,
    "dense":           dense_retriever,
    "hybrid":          hybrid_retriever,
    "hybrid_reranked": FnRetriever(hybrid_reranked_retrieve),
}
print("✅ Retrievers ready!")

Using device: cuda
GPU: Tesla T4


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

... (14 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

✅ Retrievers ready!


## 4. Generation Chain (Ollama)

In [16]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOllama(
    model="qwen2.5:3b",
    base_url="http://127.0.0.1:11434",
    temperature=0.0,
    num_ctx=8192,
)

prompt = ChatPromptTemplate.from_template(
    """Answer using ONLY the context below.
If the context does not contain the answer, say:
"I could not find that information in the provided documents."

Context:
{context}

Question: {question}

Answer:"""
)
answer_chain = prompt | llm | StrOutputParser()

def format_docs(docs):
    return "\n\n---\n\n".join(
        f"[chunk_id={d.metadata.get('chunk_id', '?')}]\n{d.page_content}"
        for d in docs
    )

def run_rag(question: str, mode: str = "hybrid_reranked"):
    if mode == "hybrid_reranked":
        docs = hybrid_reranked_retrieve(question)
    else:
        docs = RETRIEVERS[mode].invoke(question)
    answer = answer_chain.invoke({
        "context": format_docs(docs),
        "question": question,
    })
    return {"answer": answer, "n_chunks": len(docs), "docs": docs}

print("✅ Generation chain ready")

✅ Generation chain ready


### Quick smoke test

In [17]:
result = run_rag(
    "What was the role of the Military Revolutionary Committee under Leon Trotskii?",
    mode="hybrid_reranked",
)
print(result["answer"])
print(f"\n(used {result['n_chunks']} chunks)")

The Military Revolutionary Committee, appointed by the Soviet under Leon Trotsky, was tasked with organising the socialist seizure of power. This committee was formed as tensions between the Provisional Government and the Bolsheviks grew. On 16 October 1917, Lenin persuaded the Petrograd Soviet and the Bolshevik Party to agree to a socialist seizure of power, and the Military Revolutionary Committee was appointed to organise this event.

(used 5 chunks)


In [18]:
result

{'answer': 'The Military Revolutionary Committee, appointed by the Soviet under Leon Trotsky, was tasked with organising the socialist seizure of power. This committee was formed as tensions between the Provisional Government and the Bolsheviks grew. On 16 October 1917, Lenin persuaded the Petrograd Soviet and the Bolshevik Party to agree to a socialist seizure of power, and the Military Revolutionary Committee was appointed to organise this event.',
 'n_chunks': 5,
 'docs': [Document(metadata={'source': '/content/TEST_RAG/TEST_HISTORY_RAG_PDF.pdf', 'filetype': 'application/pdf', 'languages': ['eng'], 'page_number': 12, 'orig_elements': 'eJy1Vl1v2zYU/SuEny3B+rAs7S3tgGFY2gZNhj2kRUBR1zYXihRIyq5X7L/vkJLTNAs6LEDezCte3o9z7rm+/bogRT1pfye7xU9sIbK6zlddm4iqKpJStFnSbsU6qTKeV0VXtQWvF0u26MnzjnsOn68LYYztpOaeXDwrfjKjv9uT3O09LPmqquAzm4+y83tYs3LdwDoYqX3wu73NVnVaLtmmqNPi85I9nJsircO5rqp088w53odh4U7OUx/quJJfSF0PXNDib3zoyJPw0ug7obhzd4M1La6t0nrTlGtc2EpF/jRQ8H3z7iqUKN0dffGWC0+xNd6OtIhl6N3Id7HW2wXp3SKEHmC502

## 5. Evaluation Dataset
Replace the dict below with your full 50-question set.

In [19]:
import json

ground_truth_all = {int(k): v for k, v in json.loads(r'''{
  "0": {
    "question": "What did liberals in nineteenth-century Europe mean by a 'representative, elected parliamentary government,' and why were they not considered 'democrats'?",
    "answer": "Liberals argued for a representative, elected parliamentary government, subject to laws interpreted by a well-trained judiciary that was independent of rulers and officials. They were not democrats because they did not believe in universal adult franchise; they felt men of property mainly should have the vote, and they did not want the vote for women.",
    "document": "TEST_HISTORY_RAG_PDF.pdf",
    "relevant_pages": [1],
    "relevant_chunk_ids": [1, 2],
    "status": "Correct"
  },
  "1": {
    "question": "Who was Robert Owen, and what cooperative community did he attempt to build in the USA?",
    "answer": "Robert Owen (1771-1858) was a leading English manufacturer who sought to build a cooperative community called New Harmony in Indiana (USA).",
    "document": "TEST_HISTORY_RAG_PDF.pdf",
    "relevant_pages": [3],
    "relevant_chunk_ids": [5, 6],
    "status": "Correct"
  }
}''').items()}

print(f"Evaluation set size: {len(ground_truth_all)} questions")

Evaluation set size: 2 questions


## 6. Batch Retrieval + Generation

In [20]:
import pandas as pd
from tqdm.auto import tqdm

records = []
for qid, gt in tqdm(ground_truth_all.items(), desc="Questions"):
    question = gt["question"]
    reference = gt["answer"]
    relevant_ids = gt.get("relevant_chunk_ids", [])

    reference_contexts = []
    for cid in relevant_ids:
        if 0 <= cid < len(chunks):
            reference_contexts.append(chunks[cid].page_content)

    for mode, retriever in RETRIEVERS.items():
        docs = retriever.invoke(question)
        context_str = format_docs(docs)
        try:
            response = answer_chain.invoke(
                {"context": context_str, "question": question}
            )
        except Exception as e:
            response = f"[GENERATION_ERROR: {type(e).__name__}: {e}]"

        records.append({
            "qid": qid,
            "question": question,
            "mode": mode,
            "ground_truth": reference,
            "prediction": response,
            "reference_contexts": reference_contexts,
            "relevant_chunk_ids": relevant_ids,
            "retrieved_contexts": [d.page_content for d in docs],
            "retrieved_sources": [
                f"{d.metadata.get('source', '?')}#p{d.metadata.get('page', '?')}"
                for d in docs
            ],
            "n_chunks": len(docs),
        })

df_eval = pd.DataFrame(records)
print(f"✅ {len(df_eval)} rows ({len(ground_truth_all)} questions × {len(RETRIEVERS)} modes)")

Questions:   0%|          | 0/2 [00:00<?, ?it/s]

✅ 8 rows (2 questions × 4 modes)


## 7. Lightweight Metrics

In [21]:
import re
import numpy as np
from collections import Counter
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

def tokenize(t: str):
    return re.findall(r"\b\w+\b", str(t).lower())

def token_f1(pred, ref):
    p, r = Counter(tokenize(pred)), Counter(tokenize(ref))
    if not p or not r:
        return 0.0
    ov = sum((p & r).values())
    if ov == 0:
        return 0.0
    prec = ov / sum(p.values())
    rec  = ov / sum(r.values())
    return 2 * prec * rec / (prec + rec)

def exact_match(pred, ref):
    return float(str(pred).strip().lower() == str(ref).strip().lower())

# Semantic similarity model also on GPU
_embedder = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device=DEVICE,
)

def semantic_sim(a, b):
    v = _embedder.encode([str(a), str(b)], normalize_embeddings=True)
    return float(cosine_similarity([v[0]], [v[1]])[0][0])

try:
    from rouge_score import rouge_scorer
    _rouge = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
    HAVE_ROUGE = True
except ImportError:
    HAVE_ROUGE = False

try:
    import nltk
    from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
    nltk.download("punkt", quiet=True)
    nltk.download("punkt_tab", quiet=True)
    _smooth = SmoothingFunction().method1
    HAVE_BLEU = True
except ImportError:
    HAVE_BLEU = False

def score_row(pred, ref):
    out = {
        "exact_match": exact_match(pred, ref),
        "token_f1": token_f1(pred, ref),
        "semantic_sim": semantic_sim(pred, ref),
    }
    if HAVE_ROUGE:
        s = _rouge.score(ref, pred)
        out.update({
            "rouge1": s["rouge1"].fmeasure,
            "rouge2": s["rouge2"].fmeasure,
            "rougeL": s["rougeL"].fmeasure,
        })
    else:
        out.update({"rouge1": np.nan, "rouge2": np.nan, "rougeL": np.nan})

    out["bleu"] = (
        sentence_bleu([tokenize(ref)], tokenize(pred), smoothing_function=_smooth)
        if HAVE_BLEU else np.nan
    )
    return out

print("Metric helpers ready (ROUGE=", HAVE_ROUGE, ", BLEU=", HAVE_BLEU, ")")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Metric helpers ready (ROUGE= True , BLEU= True )


In [22]:
from tqdm.auto import tqdm
tqdm.pandas()

scored = df_eval.progress_apply(
    lambda r: pd.Series(score_row(r["prediction"], r["ground_truth"])),
    axis=1,
)
df_scored = pd.concat([df_eval, scored], axis=1)

METRICS = ["exact_match", "token_f1", "semantic_sim", "rouge1", "rouge2", "rougeL", "bleu"]
summary = (
    df_scored.groupby("mode")[METRICS]
    .agg(["mean", "std", "count"])
    .round(4)
)
print(summary)

  0%|          | 0/8 [00:00<?, ?it/s]

                exact_match            token_f1               semantic_sim  \
                       mean  std count     mean     std count         mean   
mode                                                                         
dense                   0.0  0.0     2   0.5562  0.0705     2       0.8913   
hybrid                  0.0  0.0     2   0.8378  0.2293     2       0.9262   
hybrid_reranked         0.0  0.0     2   0.7556  0.3457     2       0.9212   
sparse                  0.0  0.0     2   0.7417  0.2979     2       0.9132   

                               rouge1  ...        rouge2                \
                    std count    mean  ... count    mean     std count   
mode                                   ...                               
dense            0.0764     2  0.5625  ...     2  0.4159  0.2074     2   
hybrid           0.1034     2  0.8378  ...     2  0.7740  0.3197     2   
hybrid_reranked  0.1104     2  0.7667  ...     2  0.6461  0.5005     2   
sparse   

## 8. Optional LLM-as-Judge

In [35]:
# =========================================================
# 9. LLM-as-Judge (clean version)
# =========================================================

JUDGE_SYSTEM = """You are a strict evaluator of RAG answers.
Compare the PREDICTION to the GROUND TRUTH.
Scoring rubric (integers only):
  correctness: 5=fully correct, 4=mostly correct minor omissions, 3=partially correct, 2=mostly wrong, 1=wrong/unrelated.
  faithfulness: 5=entirely supported by CONTEXT, 4=supported with minor extrapolation, 3=partial support, 2=mostly unsupported, 1=contradicts context.

Return ONLY a valid JSON object like this:
{{"correctness": <int 1-5>, "faithfulness": <int 1-5>, "reason": "<short explanation>"}}

Do not write anything else. No markdown. No extra text.
Be extremely unbiased and never give 5 rating."""

JUDGE_HUMAN = """QUESTION:
{q}

GROUND TRUTH:
{gt}

CONTEXT (retrieved):
{ctx}

PREDICTION:
{pred}"""

judge_prompt = ChatPromptTemplate.from_messages([
    ("system", JUDGE_SYSTEM),
    ("human", JUDGE_HUMAN)
])
judge_chain = judge_prompt | llm | StrOutputParser()

def _fmt_ctx(row, limit_chars=4000):
    ctxs = row.get("retrieved_contexts", [])
    if not isinstance(ctxs, list):
        return ""
    text = "\n---\n".join(str(c) for c in ctxs)
    return text[:limit_chars]

def extract_json(text: str):
    """Safely extract the first JSON object from model output."""
    if not text:
        return None
    # Find the first {...} block
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if not match:
        return None
    try:
        return json.loads(match.group(0))
    except Exception:
        return None

def judge_row(row):
    try:
        raw = judge_chain.invoke({
            "q": row["question"],
            "gt": row["ground_truth"],
            "ctx": _fmt_ctx(row),
            "pred": row["prediction"],
        })
        data = extract_json(raw)
        if not data:
            return {
                "llm_correctness": np.nan,
                "llm_faithfulness": np.nan,
                "llm_reason": (raw or "")[:200]
            }
        return {
            "llm_correctness": float(data.get("correctness", np.nan)),
            "llm_faithfulness": float(data.get("faithfulness", np.nan)),
            "llm_reason": str(data.get("reason", ""))[:240]
        }
    except Exception as e:
        return {
            "llm_correctness": np.nan,
            "llm_faithfulness": np.nan,
            "llm_reason": f"err: {type(e).__name__}: {e}"[:240]
        }

print("Judge chain ready.")

Judge chain ready.


In [36]:
from tqdm.auto import tqdm
tqdm.pandas()

# Important: start from a clean df (remove old llm_ columns first)
df_base = df_scored[[c for c in df_scored.columns if not c.startswith("llm_")]].copy()

judged = df_base.progress_apply(lambda r: pd.Series(judge_row(r)), axis=1)
df_scored = pd.concat([df_base, judged], axis=1)

print(df_scored[["qid", "mode", "token_f1", "semantic_sim",
                 "llm_correctness", "llm_faithfulness", "llm_reason"]].head(10))

  0%|          | 0/8 [00:00<?, ?it/s]

   qid             mode  token_f1  semantic_sim  llm_correctness  \
0    0           sparse  0.531073      0.850989              3.0   
1    0            dense  0.506329      0.837267              5.0   
2    0           hybrid  0.675676      0.853098              5.0   
3    0  hybrid_reranked  0.511111      0.843121              4.0   
4    1           sparse  0.952381      0.975407              5.0   
5    1            dense  0.606061      0.945315              5.0   
6    1           hybrid  1.000000      0.999270              5.0   
7    1  hybrid_reranked  1.000000      0.999270              5.0   

   llm_faithfulness                                         llm_reason  
0               2.0  The prediction includes some accurate points b...  
1               5.0  The prediction accurately reflects the GROUND ...  
2               5.0  The prediction accurately reflects the ground ...  
3               4.0  The prediction accurately captures the main po...  
4               4.0  T

In [37]:
# First remove fully empty llm_ columns (handles duplicates safely)
cols = list(df_scored.columns)
keep_cols = []
seen = set()

for i, col in enumerate(cols):
    if not col.startswith("llm_"):
        keep_cols.append(col)
        continue

    # Check if this column is all NaN
    series = df_scored.iloc[:, i]
    if series.isna().all():
        continue   # drop it

    # Keep only the last non-empty occurrence of each llm_ column
    if col in seen:
        # remove previous occurrence
        keep_cols = [c for c in keep_cols if c != col]
    keep_cols.append(col)
    seen.add(col)

df_scored = df_scored[keep_cols].copy()

print(df_scored[["qid", "mode", "token_f1", "semantic_sim", "llm_correctness", "llm_faithfulness"]])

   qid             mode  token_f1  semantic_sim  llm_correctness  \
0    0           sparse  0.531073      0.850989              3.0   
1    0            dense  0.506329      0.837267              5.0   
2    0           hybrid  0.675676      0.853098              5.0   
3    0  hybrid_reranked  0.511111      0.843121              4.0   
4    1           sparse  0.952381      0.975407              5.0   
5    1            dense  0.606061      0.945315              5.0   
6    1           hybrid  1.000000      0.999270              5.0   
7    1  hybrid_reranked  1.000000      0.999270              5.0   

   llm_faithfulness  
0               2.0  
1               5.0  
2               5.0  
3               4.0  
4               4.0  
5               5.0  
6               5.0  
7               4.0  


## 9. Save Results

In [ ]:
import json
from pathlib import Path

OUT_DIR = Path("/content/rag_exports")
OUT_DIR.mkdir(parents=True, exist_ok=True)

pkl_path = OUT_DIR / "df_scored.pkl"
df_scored.to_pickle(pkl_path)
print(f"✅ Pickle → {pkl_path}")

csv_path = OUT_DIR / "df_scored.csv"
df_csv = df_scored.copy()
for col in ["retrieved_contexts", "retrieved_sources",
            "reference_contexts", "relevant_chunk_ids"]:
    if col in df_csv.columns:
        df_csv[col] = df_csv[col].apply(
            lambda x: json.dumps(x, ensure_ascii=False)
            if isinstance(x, (list, tuple)) else x
        )
df_csv.to_csv(csv_path, index=False)
print(f"✅ CSV    → {csv_path}")